# 02 — Split por grupos (60/20/20)

**Qué hace:** mueve los duplicados exactos detectados en `01_eda_y_duplicados.ipynb` a `asl_processed/deprecated/`, consolida los archivos restantes por clase, agrupa cada clase en "grupos visuales" con pHash (`group_by_phash`, para no separar variantes de la misma toma entre splits) y reparte esos grupos 60% train / 20% test / 20% validate hacia `asl_split_nuevo/`. Termina verificando dimensiones/formato del `train` resultante.

**Consume:** `asl_processed/<split>/<clase>/*` (mismo dataset revisado en el notebook 01).

**Produce:** `asl_processed/deprecated/` (duplicados movidos), `asl_split_nuevo/train|test|validate/<clase>/*` (split limpio usado por el resto de los notebooks).

> Nota de reorganización: la celda del reparto 60/20/20 usaba en el notebook original las variables `grupos` y `clase`, que no estaban definidas en esa sección — eran residuos ("variable leakage") de una celda anterior de la sección de EDA que por casualidad ya se había ejecutado antes en el mismo kernel. Aquí se corrigen a `groups` y `class_` (las variables correctas de esa misma celda) para que el cálculo de `n_test`/`n_val` use el conteo real de grupos de cada clase y el notebook sea ejecutable de forma independiente.

## Imports y rutas base

In [ ]:
from pathlib import Path
from PIL import Image
from collections import Counter, defaultdict
import hashlib

DATA_DIR = Path('./asl_dataset')
classes = sorted([folder.name for folder in DATA_DIR.iterdir() if folder.is_dir()])


## Mover duplicados exactos a `deprecated/`

In [ ]:
import shutil
### move deprecated duplicated images
DATA_DIR_DEPRECATED = Path('./asl_processed/deprecated')
DATA_DIR_PROCESSED = Path('./asl_processed')        # origen (con su train/test viejo)

In [31]:
def return_print(path):
    return hashlib.md5(path.read_bytes()).hexdigest()

In [32]:
#agrupar archivos
paths_with_duplicate = defaultdict(list)

for ruta in DATA_DIR_PROCESSED.glob("*/*/*"):
    paths_with_duplicate[return_print(ruta)].append(ruta)

In [42]:
## de cada grupo conservar el primero dentro de test

isTest = lambda path: 'test' in str(path)

DATA_DIR_DEPRECATED.mkdir(exist_ok=True)
for duplicate in paths_with_duplicate.values():
    if (len(duplicate) > 1):
        paths = sorted(duplicate, key=isTest, reverse=True)
        keep, *move = paths
        for mov in move:
            origin = f'{mov.parts[-3]}_{mov.parts[-2]}_{mov.name}'
            shutil.move(mov, str(DATA_DIR_DEPRECATED/origin))
           



## Reparto 60/20/20 por grupos visuales (pHash) hacia `asl_split_nuevo/`

In [ ]:
import random
import imagehash

# --- Configuración ---
DESTINO = Path('./asl_split_nuevo')                 # aquí construimos el split limpio
PROPORCION_TEST = 0.20                               # 20% de los GRUPOS a test
PROPORCION_VALIDATE = 0.20
UMBRAL_HAMMING = 5                                   # distancia <= 5 => mismo grupo
SEMILLA = 42                                         # reproducibilidad del reparto

random.seed(SEMILLA)   # fija el azar: el mismo split cada vez que se corra


In [47]:
## Consolidar

files_by_class = defaultdict(list)
for ruta in DATA_DIR_PROCESSED.glob("*/*/*"):        # split / clase / archivo
    clase = ruta.parts[-2]                           # el nombre de la carpeta-clase
    files_by_class[clase].append(ruta)

clases = sorted(files_by_class.keys())
print(f"Clases encontradas: {len(clases)}")

Clases encontradas: 36


In [48]:
# ============================================================
# FASE 2 — Agrupar near-duplicates por pHash, DENTRO de cada clase.
#          Cada grupo = una toma y sus variantes (unidad indivisible).
# ============================================================

def group_by_phash(group_path):
    groups = []

    for group_path in group_path:
        with Image.open(group_path) as img:
            hash = imagehash.phash(img)

        for group in groups:
            if hash - group['rep'] <= UMBRAL_HAMMING:
                group['rutas'].append(group_path)
                break

        else:
            groups.append({ 'rep': hash, 'rutas': [group_path]})

    return [group['rutas'] for group in groups]

In [51]:
# ============================================================
# FASE 3 y 4 — Repartir GRUPOS 80/20 por clase y copiar al destino.
# ============================================================

total_train = total_test = 0

for class_ in clases:
    groups = group_by_phash(files_by_class[class_])

     # Cuántos grupos van a test (al menos 1, para que ninguna clase quede sin test)
    n_test = max(1, round(len(groups) * PROPORCION_TEST))
    n_val = max(1, round(len(groups) * PROPORCION_VALIDATE))
    grupos_test = groups[:n_test]
    grupos_val = groups[n_test : n_test + n_val]
    grupos_train = groups[n_test + n_val:]

    # Copiar: cada grupo entero va a su split. Las variantes nunca se separan.

    for split, group_split in [('train', grupos_train), ('test',grupos_test), ('validate', grupos_val)]:
        folder = DESTINO / split / class_
        folder.mkdir(parents=True, exist_ok=True)

        for group in group_split:
            for path in group:
                shutil.copy2(path, folder/path.name)


    n_img_train = sum(len(g) for g in grupos_train)
    n_img_test  = sum(len(g) for g in grupos_test)
    total_train += n_img_train
    total_test  += n_img_test
    print(f"  {class_}: {len(groups)} grupos -> "
          f"train {n_img_train} img / test {n_img_test} img")

# --- Reporte final ---
total = total_train + total_test
print(f"\nTrain: {total_train} imágenes ({total_train/total:.1%})")
print(f"Test:  {total_test} imágenes ({total_test/total:.1%})")


  W: 30 grupos -> train 796 img / test 60 img
  W: 30 grupos -> train 261 img / test 368 img
  W: 30 grupos -> train 213 img / test 404 img
  W: 30 grupos -> train 195 img / test 386 img
  W: 30 grupos -> train 263 img / test 430 img
  W: 30 grupos -> train 134 img / test 528 img
  W: 30 grupos -> train 206 img / test 413 img
  W: 30 grupos -> train 145 img / test 474 img
  W: 30 grupos -> train 305 img / test 382 img
  W: 30 grupos -> train 168 img / test 464 img
  W: 30 grupos -> train 236 img / test 453 img
  W: 30 grupos -> train 241 img / test 425 img
  W: 30 grupos -> train 253 img / test 453 img
  W: 30 grupos -> train 261 img / test 457 img
  W: 30 grupos -> train 121 img / test 427 img
  W: 30 grupos -> train 224 img / test 495 img
  W: 30 grupos -> train 184 img / test 448 img
  W: 30 grupos -> train 306 img / test 286 img
  W: 30 grupos -> train 98 img / test 479 img
  W: 30 grupos -> train 65 img / test 523 img
  W: 30 grupos -> train 313 img / test 346 img
  W: 30 grupos -

## Verificación de dimensiones/formato del `train` resultante

In [63]:
#dimension, formato, tamaño, canales

size, mode, formats = Counter(), Counter(), Counter()

for class_ in classes:
    for file_ in list((DESTINO/ 'train' / class_).glob('*')):
        with Image.open(file_) as img:
            size[img.size] += 1
            mode[img.mode] += 1
            formats[file_.suffix] += 1

print(f'tamaños {size}')
print(f'modos de color {mode}')
print(f'formatos {formats}')

tamaños Counter({(134, 206): 38, (134, 205): 33, (126, 211): 32, (106, 230): 30, (104, 219): 29, (104, 220): 28, (137, 242): 27, (126, 213): 26, (137, 243): 25, (127, 249): 25, (142, 142): 25, (109, 189): 24, (118, 255): 24, (135, 205): 23, (127, 174): 23, (113, 205): 23, (136, 202): 22, (129, 255): 22, (136, 200): 21, (200, 127): 21, (142, 143): 21, (112, 204): 21, (112, 255): 20, (137, 244): 20, (166, 243): 20, (113, 206): 20, (107, 148): 20, (135, 202): 19, (112, 209): 19, (116, 164): 19, (119, 213): 19, (99, 173): 19, (121, 257): 18, (113, 212): 18, (114, 214): 18, (100, 186): 18, (113, 215): 18, (136, 203): 18, (135, 201): 18, (186, 227): 18, (143, 143): 18, (121, 225): 17, (112, 205): 17, (126, 212): 17, (164, 219): 17, (118, 213): 17, (118, 212): 17, (166, 173): 17, (136, 201): 16, (185, 227): 16, (128, 249): 16, (111, 213): 16, (114, 213): 16, (99, 186): 16, (121, 213): 16, (113, 240): 16, (105, 222): 16, (203, 238): 16, (126, 197): 16, (113, 200): 16, (105, 219): 16, (110, 189